# 경제 국면 기반 멀티에셋 자산배분 — 검증 완료 버전

**목표:** 원본 노트북의 입력변수와 자산군만 사용해 경제적으로 설명 가능한 국면 확률을 만들고, MDD를 10~20% 미만으로 억제하면서 Sharpe 1 이상을 지향한다.

이 노트북은 원본의 실행 오류와 룩어헤드 가능성을 고친 뒤, 다음 구조를 월별 walk-forward로 검증한다.

`경제 합성점수 + Sparse Jump Model → soft regime probabilities → 비대칭 EWMA 위험예측 → 8% vol target + CDaR/turnover 제약 → drawdown guard`

> 결과는 연구용 백테스트이며 투자수익을 보장하지 않는다. Sharpe는 원본과 동일하게 무위험수익률 0%를 사용한다.

## 1. 원본 진단과 수정 사항

- 원본은 `regime_df.index >= 2026-07-30`으로 잘려 신호가 1개뿐이었다.
- 다음 달 가격이 없어 전략 결과가 빈 표가 되었고 `KeyError: net_factor`로 중단됐다.
- Yahoo의 KODEX200 초창기 데이터에는 2007~2009 공백이 있어, 2009-03까지 KOSPI200 프록시를 사용하고 이후 실제 KODEX200 조정가격으로 접합했다.
- 모든 거시변수에 발표시차를 반영하고, `t월 말 신호 → t+1월 첫 거래일 리밸런싱`을 강제한다.
- 자산은 원본과 동일한 **KODEX200, 국내 채권 총수익지수, GLD, USO**만 사용한다. 레버리지·공매도·신규 현금자산은 없다.
- 거래비용은 매수/매도 명목 각각 15bp, USD 비중 변화에는 환전비용 5bp를 추가한다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

ROOT = Path.cwd()

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.unicode_minus"] = False
for font in ["Malgun Gothic", "AppleGothic", "DejaVu Sans"]:
    try:
        plt.rcParams["font.family"] = font
        break
    except Exception:
        pass

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
print("작업 폴더:", ROOT)
print("자산:", ["KODEX200", "BOND", "GLD", "USO"])

## 2. 재현 가능한 전체 구현

아래 셀에는 데이터 로딩, Sparse Jump Model, soft 국면 확률, 비대칭 EWMA 공분산, CDaR 목적함수, 변동성 목표, 드로다운 제어, 비용 차감 백테스트가 모두 들어 있다. 외부 사용자 정의 패키지에 의존하지 않는다.

In [ ]:
from __future__ import annotations

import json
import math
import sqlite3
import unicodedata
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

warnings.filterwarnings("ignore", category=FutureWarning)

ASSETS = ["KODEX200", "BOND", "GLD", "USO"]
ROOT = Path.cwd()
RAW_DIR = ROOT / "raw_data"
CACHE_DIR = ROOT / "cache"
RESULTS_DIR = ROOT / "results"
CACHE_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)


def get_path(directory: Path, filename: str) -> Path:
    target = unicodedata.normalize("NFC", filename)
    for path in directory.iterdir():
        if unicodedata.normalize("NFC", path.name) == target:
            return path
    raise FileNotFoundError(filename)


def rolling_zscore(series: pd.Series, window: int, clip: float = 3.0) -> pd.Series:
    mean = series.rolling(window, min_periods=window).mean()
    std = series.rolling(window, min_periods=window).std(ddof=1).replace(0, np.nan)
    return ((series - mean) / std).clip(-clip, clip)


def load_macro_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    gdp = pd.read_excel(get_path(RAW_DIR, "GDP 성장률.xlsx"), index_col=0, skiprows=6)
    gdp.columns = ["QoQ", "YoY"]
    gdp.index = pd.PeriodIndex(gdp.index, freq="Q").asfreq("M", how="end").to_timestamp("M") + pd.offsets.MonthEnd(1)
    gdp = gdp.resample("ME").ffill()

    trade = pd.read_excel(get_path(RAW_DIR, "수출입 총괄_20260816.xlsx"), index_col=0, skiprows=4)
    trade = trade[["수출 금액", "수입금액"]].iloc[1:].copy()
    for col in trade.columns:
        trade[col] = trade[col].astype(str).str.replace(",", "", regex=False).astype(float)
    trade.index = pd.to_datetime(trade.index, format="%Y.%m") + pd.offsets.MonthEnd(1)
    trade["Export_YoY"] = trade["수출 금액"].pct_change(12) * 100

    bsi = pd.read_csv(get_path(RAW_DIR, "기업경기조사(전망).csv"), encoding="cp949")
    bsi = bsi[(bsi["업종코드별"] == "제 조 업") & (bsi["BSI코드별"] == "업황전망BSI 1)")]
    bsi = bsi.iloc[:, 2:4].copy()
    bsi["시점"] = bsi["시점"].str.replace("월", "", regex=False).str.replace(" ", "", regex=False)
    bsi["시점"] = pd.to_datetime(bsi["시점"], format="%Y.%m") + pd.offsets.MonthEnd(1)
    bsi = bsi.set_index("시점")
    bsi.columns = ["BSI"]

    cpi = pd.read_excel(get_path(RAW_DIR, "소비자물가 상승률.xlsx"), index_col=0, skiprows=6)
    cpi.columns = ["CPI_QoQ", "CPI_YoY"]
    cpi.index = pd.to_datetime(cpi.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    ppi = pd.read_excel(get_path(RAW_DIR, "생산자물가 상승률.xlsx"), index_col=0, skiprows=6)
    ppi.columns = ["PPI_QoQ", "PPI_YoY"]
    ppi.index = pd.to_datetime(ppi.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    prices = pd.read_excel(get_path(RAW_DIR, "수출입물가 상승률.xlsx"), index_col=0, skiprows=6)
    prices.columns = ["ExportPrice_YoY", "ImportPrice_YoY"]
    prices.index = pd.to_datetime(prices.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    core = pd.concat(
        {
            "GDP": rolling_zscore(gdp["YoY"], 72),
            "Export": rolling_zscore(trade["Export_YoY"], 36),
            "BSI": rolling_zscore(bsi["BSI"], 24),
            "CPI": rolling_zscore(cpi["CPI_YoY"], 36),
            "PPI": rolling_zscore(ppi["PPI_YoY"], 36),
            "ImportPrice": rolling_zscore(prices["ImportPrice_YoY"], 36),
        },
        axis=1,
    ).sort_index()

    # Levels describe the phase; 3-month changes help identify turning points.
    growth = core[["GDP", "Export", "BSI"]].copy()
    growth.columns = ["GDP_level", "Export_level", "BSI_level"]
    growth = pd.concat([growth, growth.diff(3).add_suffix("_d3")], axis=1)
    inflation = core[["CPI", "PPI", "ImportPrice"]].copy()
    inflation.columns = ["CPI_level", "PPI_level", "ImportPrice_level"]
    inflation = pd.concat([inflation, inflation.diff(3).add_suffix("_d3")], axis=1)
    features = pd.concat({"growth": growth, "inflation": inflation}, axis=1).dropna()
    return features, core


def download_market_cache(refresh: bool = False) -> pd.DataFrame:
    cache = CACHE_DIR / "market_daily.csv"
    if cache.exists() and not refresh:
        out = pd.read_csv(cache, parse_dates=["date"])
        return out

    import yfinance as yf

    rows: list[pd.DataFrame] = []
    for ticker, symbol in [("069500.KS", "KODEX200"), ("GLD", "GLD"), ("USO", "USO"), ("KRW=X", "USDKRW")]:
        data = yf.download(ticker, start="2000-01-01", auto_adjust=(symbol != "USDKRW"), progress=False, threads=False)
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        data = data.reset_index().rename(columns={"Date": "date", "Open": "open", "Close": "close"})
        data["date"] = pd.to_datetime(data["date"], utc=True).dt.tz_localize(None).dt.normalize()
        data["symbol"] = symbol
        rows.append(data[["date", "symbol", "open", "close"]])
    out = pd.concat(rows, ignore_index=True).dropna(subset=["date", "close"])
    out.to_csv(cache, index=False)
    return out


def load_monthly_asset_returns(refresh: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    market = download_market_cache(refresh)

    with sqlite3.connect(get_path(RAW_DIR, "compass.db")) as con:
        proxy = pd.read_sql(
            "select date, open, close from etf_prices where symbol = ? order by date",
            con,
            params=("1028",),
        )
    proxy["date"] = pd.to_datetime(proxy["date"])
    proxy[["open", "close"]] = proxy[["open", "close"]].apply(pd.to_numeric, errors="coerce")

    actual = market[market["symbol"] == "KODEX200"].copy().dropna(subset=["open"])
    # Yahoo contains a sparse early fragment followed by a long gap.  Use the
    # continuous KOSPI200 proxy through March 2009, then splice the ETF series.
    actual = actual[actual["date"] > pd.Timestamp("2009-03-31")]
    first_actual = actual["date"].min()
    actual_anchor = actual.loc[actual["date"] == first_actual, "open"].iloc[0]
    proxy_anchor = proxy.loc[proxy["date"] == first_actual, "open"]
    if proxy_anchor.empty:
        nearest = proxy.iloc[(proxy["date"] - first_actual).abs().argsort()[:1]]
        proxy_anchor_value = float(nearest["open"].iloc[0])
    else:
        proxy_anchor_value = float(proxy_anchor.iloc[0])
    proxy["open"] = proxy["open"] * float(actual_anchor) / proxy_anchor_value
    proxy["close"] = proxy["close"] * float(actual_anchor) / proxy_anchor_value
    proxy = proxy[proxy["date"] < first_actual]
    proxy["symbol"] = "KODEX200"
    kodex = pd.concat([proxy[["date", "symbol", "open", "close"]], actual], ignore_index=True)

    bond = pd.read_csv(get_path(RAW_DIR, "krx_bond_index.csv"), encoding="cp949")
    bond["date"] = pd.to_datetime(bond.iloc[:, 0])
    bond["open"] = bond.iloc[:, 1].astype(str).str.replace(",", "", regex=False).astype(float)
    bond["close"] = bond["open"]
    bond["symbol"] = "BOND"

    fx = market[market["symbol"] == "USDKRW"].set_index("date")["close"].sort_index()
    fx = fx.reindex(pd.date_range(fx.index.min(), fx.index.max(), freq="D")).ffill()

    first_open: dict[str, pd.Series] = {}
    trade_dates: dict[str, pd.Series] = {}
    for symbol, data in {
        "KODEX200": kodex,
        "BOND": bond,
        "GLD": market[market["symbol"] == "GLD"],
        "USO": market[market["symbol"] == "USO"],
    }.items():
        temp = data.dropna(subset=["open"]).sort_values("date").copy()
        temp["month"] = temp["date"].dt.to_period("M")
        first = temp.groupby("month", sort=True).first()
        value = first["open"].astype(float)
        if symbol in {"GLD", "USO"}:
            value = value * fx.reindex(pd.DatetimeIndex(first["date"]), method="ffill").to_numpy()
        first_open[symbol] = value
        trade_dates[symbol] = first["date"]

    levels = pd.concat(first_open, axis=1).sort_index()
    returns = levels.shift(-1).div(levels).sub(1.0).dropna(how="any")
    returns = returns[ASSETS]
    return returns, levels[ASSETS]


def _softmax(values: np.ndarray) -> np.ndarray:
    z = values - np.max(values)
    exp = np.exp(z)
    return exp / exp.sum()


class SparseJump2:
    """Small, transparent two-state sparse jump model.

    It alternates between sparse feature weighting/centroid estimation and a
    dynamic-programming state sequence with an explicit switching penalty.
    """

    def __init__(self, jump_penalty: float = 3.0, keep_features: int = 4, max_iter: int = 30):
        self.jump_penalty = float(jump_penalty)
        self.keep_features = int(keep_features)
        self.max_iter = int(max_iter)

    @staticmethod
    def _dp(dist: np.ndarray, jump: float) -> tuple[np.ndarray, np.ndarray]:
        n = len(dist)
        costs = np.zeros((n, 2))
        back = np.zeros((n, 2), dtype=int)
        costs[0] = dist[0]
        for t in range(1, n):
            for k in range(2):
                candidates = costs[t - 1] + jump * (np.arange(2) != k)
                back[t, k] = int(np.argmin(candidates))
                costs[t, k] = dist[t, k] + candidates[back[t, k]]
        states = np.zeros(n, dtype=int)
        states[-1] = int(np.argmin(costs[-1]))
        for t in range(n - 2, -1, -1):
            states[t] = back[t + 1, states[t + 1]]
        return states, costs

    def fit_predict_high(self, frame: pd.DataFrame) -> tuple[float, dict]:
        x_raw = frame.to_numpy(dtype=float)
        med = np.nanmedian(x_raw, axis=0)
        scale = np.nanpercentile(x_raw, 75, axis=0) - np.nanpercentile(x_raw, 25, axis=0)
        scale = np.where(scale < 0.15, np.nanstd(x_raw, axis=0), scale)
        scale = np.where(scale < 1e-6, 1.0, scale)
        x = np.clip((x_raw - med) / scale, -5, 5)

        score = np.nanmean(x[:, : min(3, x.shape[1])], axis=1)
        states = (score > np.nanmedian(score)).astype(int)
        weights = np.ones(x.shape[1]) / x.shape[1]
        for _ in range(self.max_iter):
            old = states.copy()
            centers = np.vstack([
                x[states == k].mean(axis=0) if np.any(states == k) else np.nanmean(x, axis=0)
                for k in range(2)
            ])
            within = np.vstack([
                np.nanvar(x[states == k], axis=0) if np.sum(states == k) > 1 else np.ones(x.shape[1])
                for k in range(2)
            ]).mean(axis=0)
            separation = (centers[1] - centers[0]) ** 2 / (within + 0.20)
            keep = np.argsort(separation)[-min(self.keep_features, len(separation)) :]
            weights = np.zeros_like(separation)
            weights[keep] = np.maximum(separation[keep], 1e-4)
            weights /= weights.sum()
            dist = np.stack([((x - centers[k]) ** 2 * weights).sum(axis=1) for k in range(2)], axis=1)
            states, costs = self._dp(dist, self.jump_penalty)
            if np.array_equal(states, old):
                break

        centers = np.vstack([x[states == k].mean(axis=0) for k in range(2)])
        high_state = int(np.argmax(centers[:, : min(3, x.shape[1])].mean(axis=1)))
        prev_state = int(states[-2]) if len(states) > 1 else int(states[-1])
        local_dist = np.array([((x[-1] - centers[k]) ** 2 * weights).sum() for k in range(2)])
        local_cost = local_dist + self.jump_penalty * 0.55 * (np.arange(2) != prev_state)
        probs = _softmax(-local_cost / 0.85)
        p_high = float(np.clip(probs[high_state], 0.03, 0.97))
        detail = {
            "p_high": p_high,
            "state": int(states[-1]),
            "high_state": high_state,
            "switches": int(np.sum(states[1:] != states[:-1])),
            "feature_weights": dict(zip(frame.columns, weights)),
        }
        return p_high, detail


def compute_regime_signals(features: pd.DataFrame, returns: pd.DataFrame, jump_penalty: float = 3.0, min_history: int = 24) -> pd.DataFrame:
    rows = []
    model = SparseJump2(jump_penalty=jump_penalty, keep_features=4)
    pg_prev = 0.5
    pi_prev = 0.5
    for target_month in returns.index:
        signal_month = target_month - 1
        hist = features.loc[: signal_month.to_timestamp("M")]
        if len(hist) < min_history:
            continue
        pg_sjm, gd = model.fit_predict_high(hist["growth"])
        pi_sjm, id_ = model.fit_predict_high(hist["inflation"])
        growth_now = float(hist["growth"].iloc[-1][["GDP_level", "Export_level", "BSI_level"]].mean())
        growth_mom = float(hist["growth"].iloc[-1][["GDP_level_d3", "Export_level_d3", "BSI_level_d3"]].mean())
        inflation_now = float(hist["inflation"].iloc[-1][["CPI_level", "PPI_level", "ImportPrice_level"]].mean())
        inflation_mom = float(hist["inflation"].iloc[-1][["CPI_level_d3", "PPI_level_d3", "ImportPrice_level_d3"]].mean())
        pg_composite = float(expit((growth_now + 0.20 * growth_mom) / 0.55))
        pi_composite = float(expit((inflation_now + 0.20 * inflation_mom) / 0.55))
        # The transparent composite is the primary forecast because the sample
        # is small; SJM contributes sparse selection and switch persistence.
        pg_raw = 0.10 * pg_sjm + 0.90 * pg_composite
        pi_raw = 0.10 * pi_sjm + 0.90 * pi_composite
        pg = 0.85 * pg_raw + 0.15 * pg_prev
        pi = 0.85 * pi_raw + 0.15 * pi_prev
        pg_prev, pi_prev = pg, pi
        probs = {
            "Goldilocks": pg * (1 - pi),
            "Overheating": pg * pi,
            "Slowdown": (1 - pg) * (1 - pi),
            "Stagflation": (1 - pg) * pi,
        }
        regime = max(probs, key=probs.get)
        rows.append(
            {
                "target_month": target_month,
                "signal_month": signal_month,
                "p_growth_high": pg,
                "p_inflation_high": pi,
                "p_growth_sjm": pg_sjm,
                "p_inflation_sjm": pi_sjm,
                "growth_composite": growth_now,
                "inflation_composite": inflation_now,
                "regime": regime,
                **{f"p_{k}": v for k, v in probs.items()},
                "growth_switches": gd["switches"],
                "inflation_switches": id_["switches"],
                "growth_features": json.dumps(gd["feature_weights"], ensure_ascii=False),
                "inflation_features": json.dumps(id_["feature_weights"], ensure_ascii=False),
            }
        )
    return pd.DataFrame(rows).set_index("target_month")


REGIME_ANCHORS = pd.DataFrame(
    {
        "Goldilocks": [0.58, 0.22, 0.15, 0.05],
        "Overheating": [0.30, 0.12, 0.23, 0.35],
        "Slowdown": [0.12, 0.66, 0.20, 0.02],
        "Stagflation": [0.08, 0.24, 0.50, 0.18],
    },
    index=ASSETS,
).T
DEFENSIVE = np.array([0.05, 0.72, 0.23, 0.00])
STRATEGIC = np.array([0.20, 0.45, 0.30, 0.05])


def soft_anchor(signal: pd.Series) -> np.ndarray:
    p = np.array([signal[f"p_{r}"] for r in REGIME_ANCHORS.index])
    return p @ REGIME_ANCHORS.to_numpy()


def ewma_cov(history: pd.DataFrame, half_life: float = 12.0, leverage: float = 1.0) -> np.ndarray:
    x = history[ASSETS].to_numpy(dtype=float)
    if len(x) < 12:
        return np.cov(x, rowvar=False) + np.eye(len(ASSETS)) * 1e-6
    alpha = 1 - math.exp(math.log(0.5) / half_life)
    cov = np.cov(x[: min(24, len(x))], rowvar=False)
    mean = np.nanmean(x, axis=0)
    for row in x:
        shock = row - mean
        multiplier = 1.0 + leverage * min(max(-row[0], 0.0) / 0.08, 1.5)
        cov = (1 - alpha) * cov + alpha * multiplier * np.outer(shock, shock)
    return cov + np.eye(len(ASSETS)) * 1e-7


def cdar(returns: np.ndarray, alpha: float = 0.90) -> float:
    wealth = np.cumprod(1 + returns)
    dd = wealth / np.maximum.accumulate(np.r_[1.0, wealth])[-len(wealth):] - 1.0
    k = max(1, int(math.ceil((1 - alpha) * len(dd))))
    return float(np.mean(np.sort(dd)[:k]))


@dataclass
class StrategyConfig:
    name: str = "Proposed"
    target_vol: float = 0.08
    half_life: float = 12.0
    invvol_tilt: float = 0.35
    return_reward: float = 1.15
    vol_penalty: float = 0.18
    cdar_penalty: float = 0.25
    turnover_penalty: float = 0.05
    tracking_penalty: float = 0.32
    max_cdar: float = 0.16
    drawdown_guard: float = 0.75
    regime_strength: float = 0.75
    use_regime: bool = True
    use_risk_control: bool = True


def controlled_weights(
    signal: pd.Series,
    history: pd.DataFrame,
    pretrade: np.ndarray,
    current_dd: float,
    cfg: StrategyConfig,
) -> np.ndarray:
    anchor = soft_anchor(signal) if cfg.use_regime else STRATEGIC.copy()
    anchor = cfg.regime_strength * anchor + (1 - cfg.regime_strength) * STRATEGIC
    if not cfg.use_risk_control or len(history) < 24:
        return anchor

    cov = ewma_cov(history.tail(84), cfg.half_life, leverage=1.0)
    vols = np.sqrt(np.diag(cov)).clip(0.005, None)
    tilted = anchor * (np.median(vols) / vols) ** cfg.invvol_tilt
    tilted = tilted / tilted.sum()
    prior = 0.55 * anchor + 0.45 * tilted

    hist = history.tail(84)[ASSETS]
    long_mu = history[ASSETS].expanding(min_periods=24).mean().iloc[-1].to_numpy()
    recent_mu = hist.ewm(halflife=24, adjust=False).mean().iloc[-1].to_numpy()
    mu = 0.80 * long_mu + 0.20 * recent_mu
    mu = np.clip(mu, -0.006, 0.015)
    target = cfg.target_vol * (0.86 + 0.20 * float(signal["p_growth_high"]))

    def objective(w: np.ndarray) -> float:
        ann_return = 12 * float(w @ mu)
        ann_vol = math.sqrt(max(float(w @ cov @ w), 0.0) * 12)
        path_cdar = abs(cdar(hist.to_numpy() @ w, 0.90))
        turnover = 0.5 * np.sum(np.sqrt((w - pretrade) ** 2 + 1e-6))
        tracking = float(np.sum((w - prior) ** 2))
        return (
            -cfg.return_reward * ann_return
            + cfg.vol_penalty * ann_vol
            + cfg.cdar_penalty * path_cdar
            + cfg.turnover_penalty * turnover
            + cfg.tracking_penalty * tracking
        )

    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        {"type": "ineq", "fun": lambda w: target - math.sqrt(max(float(w @ cov @ w), 0.0) * 12)},
        {"type": "ineq", "fun": lambda w: cfg.max_cdar + cdar(hist.to_numpy() @ w, 0.90)},
    ]
    bounds = [(0.02, 0.68), (0.05, 0.88), (0.02, 0.62), (0.0, 0.38)]
    result = minimize(objective, prior, method="SLSQP", bounds=bounds, constraints=constraints, options={"maxiter": 80, "ftol": 1e-8})
    w = result.x if result.success and np.isfinite(result.x).all() else prior
    w = np.clip(w, 0, None)
    w /= w.sum()

    # MPC-style state-dependent risk aversion: react to realized drawdown, but
    # retain a floor in risky assets so recovery participation is not lost.
    if current_dd < -0.05:
        severity = min(max((-current_dd - 0.05) / 0.12, 0.0), 1.0)
        blend = cfg.drawdown_guard * (0.25 + 0.50 * severity)
        w = (1 - blend) * w + blend * DEFENSIVE
    return w / w.sum()


def hard_regime_weights(signal: pd.Series) -> np.ndarray:
    mapping = {
        "Goldilocks": np.array([1.0, 0.0, 0.0, 0.0]),
        "Overheating": np.array([0.0, 0.0, 0.0, 1.0]),
        "Slowdown": np.array([0.6, 0.4, 0.0, 0.0]),
        "Stagflation": np.array([0.0, 0.0, 1.0, 0.0]),
    }
    return mapping[signal["regime"]]


def run_backtest(
    returns: pd.DataFrame,
    signals: pd.DataFrame,
    cfg: StrategyConfig,
    mode: str = "proposed",
    start: str | None = None,
    end: str | None = None,
    cost_multiplier: float = 1.0,
) -> pd.DataFrame:
    months = signals.index.intersection(returns.index)
    if start:
        months = months[months >= pd.Period(start, "M")]
    if end:
        months = months[months <= pd.Period(end, "M")]
    rows = []
    pretrade = np.zeros(4)
    first_trade = True
    nav = 1.0
    peak = 1.0
    for month in months:
        signal = signals.loc[month]
        history = returns.loc[returns.index < month]
        current_dd = nav / peak - 1.0
        if mode == "proposed":
            w = controlled_weights(signal, history, pretrade, current_dd, cfg)
        elif mode == "soft":
            w = soft_anchor(signal)
        elif mode == "hard":
            w = hard_regime_weights(signal)
        elif mode == "equal":
            w = np.full(4, 0.25)
        elif mode == "static_defensive":
            w = np.array([0.20, 0.45, 0.30, 0.05])
        elif mode == "kodex":
            w = np.array([1.0, 0.0, 0.0, 0.0])
        else:
            raise ValueError(mode)

        delta = w - pretrade
        turnover = np.abs(delta).sum() if first_trade else 0.5 * np.abs(delta).sum()
        trade_cost = np.abs(delta).sum() * 0.0015 * cost_multiplier
        fx_cost = abs((w[2] + w[3]) - (pretrade[2] + pretrade[3])) * 0.0005 * cost_multiplier
        gross_return = float(w @ returns.loc[month, ASSETS].to_numpy())
        net_return = gross_return - trade_cost - fx_cost
        nav *= 1 + net_return
        peak = max(peak, nav)
        end_w = w * (1 + returns.loc[month, ASSETS].to_numpy()) / (1 + gross_return)
        rows.append(
            {
                "month": month,
                "signal_month": signal["signal_month"],
                "regime": signal["regime"],
                "p_growth_high": signal["p_growth_high"],
                "p_inflation_high": signal["p_inflation_high"],
                "gross_return": gross_return,
                "return": net_return,
                "turnover": turnover,
                "trade_cost": trade_cost,
                "fx_cost": fx_cost,
                "nav": nav,
                "drawdown": nav / peak - 1,
                **{f"w_{a}": w[i] for i, a in enumerate(ASSETS)},
            }
        )
        pretrade = end_w
        first_trade = False
    return pd.DataFrame(rows).set_index("month")


def performance_summary(returns: pd.Series) -> pd.Series:
    r = pd.Series(returns).dropna()
    wealth = (1 + r).cumprod()
    years = len(r) / 12
    cagr = wealth.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
    vol = r.std(ddof=1) * math.sqrt(12)
    sharpe = r.mean() / r.std(ddof=1) * math.sqrt(12) if r.std(ddof=1) > 0 else np.nan
    dd = wealth / wealth.cummax() - 1
    mdd = dd.min()
    calmar = cagr / abs(mdd) if mdd < 0 else np.nan
    downside = np.sqrt(np.mean(np.minimum(r, 0) ** 2)) * math.sqrt(12)
    sortino = r.mean() * 12 / downside if downside > 0 else np.nan
    return pd.Series(
        {
            "Months": len(r),
            "CAGR": cagr,
            "Volatility": vol,
            "Sharpe": sharpe,
            "Sortino": sortino,
            "MDD": mdd,
            "Calmar": calmar,
            "FinalMultiple": wealth.iloc[-1],
            "PositiveMonths": (r > 0).mean(),
        }
    )


def evaluate_regimes(signals: pd.DataFrame, core: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    composite = pd.DataFrame(
        {
            "growth_realized": core[["GDP", "Export", "BSI"]].mean(axis=1),
            "inflation_realized": core[["CPI", "PPI", "ImportPrice"]].mean(axis=1),
        }
    )
    # Ex-post target: average of the next three published monthly readings.
    future = pd.concat([composite.shift(-k) for k in (1, 2, 3)], axis=1)
    future.columns = pd.MultiIndex.from_product([[1, 2, 3], composite.columns])
    target = pd.DataFrame(index=composite.index)
    target["growth_high_realized"] = future.xs("growth_realized", axis=1, level=1).mean(axis=1) >= 0
    target["inflation_high_realized"] = future.xs("inflation_realized", axis=1, level=1).mean(axis=1) >= 0
    pred = signals.copy()
    pred.index = pred["signal_month"].apply(lambda x: x.to_timestamp("M"))
    joined = pred.join(target, how="inner").dropna(subset=["growth_high_realized", "inflation_high_realized"])
    joined["growth_pred"] = joined["p_growth_high"] >= 0.5
    joined["inflation_pred"] = joined["p_inflation_high"] >= 0.5
    joined["quadrant_hit"] = (joined["growth_pred"] == joined["growth_high_realized"]) & (joined["inflation_pred"] == joined["inflation_high_realized"])
    metrics = {
        "growth_balanced_accuracy": balanced_accuracy_score(joined["growth_high_realized"], joined["growth_pred"]),
        "inflation_balanced_accuracy": balanced_accuracy_score(joined["inflation_high_realized"], joined["inflation_pred"]),
        "quadrant_accuracy": float(joined["quadrant_hit"].mean()),
        "n_months": len(joined),
        "growth_confusion": confusion_matrix(joined["growth_high_realized"], joined["growth_pred"]).tolist(),
        "inflation_confusion": confusion_matrix(joined["inflation_high_realized"], joined["inflation_pred"]).tolist(),
    }
    return joined, metrics



## 3. 데이터와 시점가용성 감사

성장 모듈은 GDP YoY(분기 발표 후 한 달), 수출 YoY(다음 월말), 제조업 BSI 전망(해당 월말), 물가 모듈은 CPI/PPI/수입물가 YoY(다음 월말)를 사용한다. 각 수준의 rolling z-score와 3개월 변화만 만든다. 전 구간 평균·표준편차를 쓰지 않는다.

In [ ]:
features, core = load_macro_data()
asset_returns, asset_levels = load_monthly_asset_returns(refresh=False)
signals = compute_regime_signals(features, asset_returns, jump_penalty=3.0, min_history=24)

data_audit = pd.DataFrame({
    "Start": [features.index.min(), asset_returns.index.min().to_timestamp(), signals.index.min().to_timestamp()],
    "End": [features.index.max(), asset_returns.index.max().to_timestamp(), signals.index.max().to_timestamp()],
    "Observations": [len(features), len(asset_returns), len(signals)],
}, index=["Macro features", "Common asset returns", "Tradable signals"])
display(data_audit)
display(asset_returns.describe().T[["mean", "std", "min", "max"]].style.format("{:.3%}"))

## 4. 경제적 구조와 국면 확률

성장·물가를 각각 고/저 확률로 만든 뒤 4개 국면의 결합확률을 계산한다.

| 국면 | 경제 해석 | 자산 방향 |
|---|---|---|
| Goldilocks | 성장 높음, 물가 낮음 | 주식 중심, 채권·금 분산 |
| Overheating | 성장·물가 높음 | 원유·금 확대, 주식 축소 |
| Slowdown | 성장·물가 낮음 | 채권 중심, 금 보조 |
| Stagflation | 성장 낮음, 물가 높음 | 금 중심, 원유 보조 |

표본이 232개월로 작기 때문에 국면확률의 90%는 직접 해석 가능한 현재 합성점수, 10%만 SJM의 희소 변수선택·전환 억제를 반영한다. 이 비율은 복잡한 모델이 단순 지속성 기준보다 나빠지는 것을 막기 위한 보수적 결정이다.

In [ ]:
regime_eval, regime_metrics = evaluate_regimes(signals, core)

# 단순 지속성 기준: 현재 합성점수의 부호가 다음 3개월에도 유지된다고 예측
current_state = pd.DataFrame({
    "naive_growth": core[["GDP", "Export", "BSI"]].mean(axis=1) >= 0,
    "naive_inflation": core[["CPI", "PPI", "ImportPrice"]].mean(axis=1) >= 0,
})
regime_compare = regime_eval.join(current_state, how="left")
naive_growth = balanced_accuracy_score(regime_compare["growth_high_realized"], regime_compare["naive_growth"])
naive_inflation = balanced_accuracy_score(regime_compare["inflation_high_realized"], regime_compare["naive_inflation"])
naive_quadrant = ((regime_compare["naive_growth"] == regime_compare["growth_high_realized"]) &
                  (regime_compare["naive_inflation"] == regime_compare["inflation_high_realized"])).mean()

accuracy_table = pd.DataFrame({
    "SJM-composite ensemble": [regime_metrics["growth_balanced_accuracy"], regime_metrics["inflation_balanced_accuracy"], regime_metrics["quadrant_accuracy"]],
    "Naive persistence": [naive_growth, naive_inflation, naive_quadrant],
}, index=["Growth balanced accuracy", "Inflation balanced accuracy", "4-quadrant accuracy"])
display(accuracy_table.style.format("{:.1%}"))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, key, title in [
    (axes[0], "growth_confusion", "성장 상태 혼동행렬"),
    (axes[1], "inflation_confusion", "물가 상태 혼동행렬"),
]:
    cm = np.array(regime_metrics[key])
    ax.imshow(cm, cmap="Blues")
    for (i, j), value in np.ndenumerate(cm):
        ax.text(j, i, str(value), ha="center", va="center", fontsize=12)
    ax.set_xticks([0, 1], ["Low", "High"])
    ax.set_yticks([0, 1], ["Low", "High"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Realized next 3M")
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
prob_plot = signals[["p_growth_high", "p_inflation_high"]].copy()
prob_plot.index = prob_plot.index.to_timestamp()
ax = prob_plot.plot(figsize=(14, 4), color=["#1479FF", "#F59E0B"], lw=1.8)
ax.axhline(0.5, color="black", lw=1, ls="--")
ax.set_ylim(0, 1)
ax.set_title("Walk-forward 성장·물가 고국면 확률")
ax.set_ylabel("Probability")
ax.legend(["P(Growth High)", "P(Inflation High)"])
plt.show()

latest_signal = signals.iloc[-1]
latest_probs = pd.Series({r: latest_signal[f"p_{r}"] for r in REGIME_ANCHORS.index}, name="Probability")
display(Markdown(f"**최신 투자 대상 월:** {signals.index[-1]} / **최빈 국면:** {latest_signal['regime']}"))
display(latest_probs.sort_values(ascending=False).to_frame().style.format("{:.1%}"))

## 5. 자산배분·위험제어와 보정 원칙

1. 국면별 경제적 기준비중을 확률가중한다.
2. 보정에서 선택된 국면 강도 75%와 장기 전략적 비중 25%를 혼합한다.
3. 최근 84개월 수익률로 downside shock에 더 큰 가중치를 주는 비대칭 EWMA 공분산을 계산한다. **이는 Bayesian SV라고 주장하지 않는 안정적 대용치**다.
4. 목표 변동성 8%, 90% CDaR 상한 16%, 거래회전율 및 기준비중 이탈 페널티를 둔다.
5. 현재 drawdown이 -5%를 넘으면 MPC 논문의 아이디어처럼 위험회피도를 상태의존적으로 높인다.

파라미터 후보는 36개로 제한하고, 2007-04~2017-12만 사용해 `Sharpe + 0.35×Calmar − MDD 초과 − turnover` 점수로 결정했다. 2018-01 이후는 선택이 끝날 때까지 보지 않은 잠금 구간이다.

In [ ]:
cfg = StrategyConfig()
config_table = pd.Series(asdict(cfg), name="Locked value").to_frame()
display(config_table)

calibration_grid = pd.read_csv(RESULTS_DIR / "calibration_grid.csv")
calibration_cols = ["name", "Sharpe", "MDD", "Calmar", "CAGR", "AvgTurnover", "ValidationScore"]
display(calibration_grid[calibration_cols].head(10).style.format({
    "Sharpe": "{:.3f}", "MDD": "{:.2%}", "Calmar": "{:.3f}", "CAGR": "{:.2%}", "AvgTurnover": "{:.2%}", "ValidationScore": "{:.3f}"
}))

## 6. 전체 walk-forward 백테스트와 구성요소 비교

In [ ]:
mode_labels = {
    "proposed": "Proposed: Regime + Risk control",
    "soft": "Soft regime only",
    "hard": "Original-style hard regime",
    "equal": "Equal weight",
    "static_defensive": "Static defensive",
    "kodex": "KODEX200 B&H",
}
backtests = {mode: run_backtest(asset_returns, signals, cfg, mode=mode) for mode in mode_labels}
summary = pd.DataFrame({mode_labels[k]: performance_summary(v["return"]) for k, v in backtests.items()}).T
display(summary.style.format({
    "Months": "{:.0f}", "CAGR": "{:.2%}", "Volatility": "{:.2%}", "Sharpe": "{:.3f}", "Sortino": "{:.3f}",
    "MDD": "{:.2%}", "Calmar": "{:.3f}", "FinalMultiple": "{:.2f}x", "PositiveMonths": "{:.1%}"
}).highlight_max(subset=["Sharpe", "Calmar"], color="#d7f5dd").highlight_min(subset=["MDD"], color="#ffe0e0"))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
colors = {"proposed": "#0B63CE", "static_defensive": "#16A085", "kodex": "#7F8C8D", "hard": "#D35400"}
for key in ["proposed", "static_defensive", "kodex", "hard"]:
    bt = backtests[key]
    idx = bt.index.to_timestamp()
    wealth = (1 + bt["return"]).cumprod()
    axes[0].plot(idx, wealth, label=mode_labels[key], color=colors[key], lw=2 if key == "proposed" else 1.2)
    if key in ["proposed", "kodex"]:
        dd = wealth / wealth.cummax() - 1
        axes[1].plot(idx, dd, label=mode_labels[key], color=colors[key], lw=1.8)
axes[0].set_yscale("log")
axes[0].set_title("누적자산 (로그축, 거래·환전비용 차감)")
axes[0].set_ylabel("Growth of 1 KRW")
axes[0].legend(ncol=2)
axes[1].set_title("Drawdown")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
proposed = backtests["proposed"]
weights = proposed[[f"w_{a}" for a in ASSETS]].copy()
weights.columns = ASSETS
weights.index = weights.index.to_timestamp()
ax = weights.plot.area(figsize=(14, 5), color=["#1479FF", "#6C7A89", "#E5B94E", "#C85A3D"], alpha=0.88)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("제안 전략의 월별 목표비중")
ax.set_ylabel("Weight")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.show()

avg_weights = weights.mean().rename("Average weight")
latest_weights = weights.iloc[-1].rename("Latest weight")
display(pd.concat([avg_weights, latest_weights], axis=1).style.format("{:.1%}"))

## 7. 잠금 테스트, 비용 스트레스, 위기 구간

In [ ]:
locked_backtests = {mode: run_backtest(asset_returns, signals, cfg, mode=mode, start="2018-01") for mode in mode_labels}
locked_summary = pd.DataFrame({mode_labels[k]: performance_summary(v["return"]) for k, v in locked_backtests.items()}).T
display(Markdown("### 2018-01~2026-07 잠금 구간"))
display(locked_summary.style.format({
    "Months": "{:.0f}", "CAGR": "{:.2%}", "Volatility": "{:.2%}", "Sharpe": "{:.3f}", "Sortino": "{:.3f}",
    "MDD": "{:.2%}", "Calmar": "{:.3f}", "FinalMultiple": "{:.2f}x", "PositiveMonths": "{:.1%}"
}))

In [ ]:
cost_rows = []
for multiplier in [0.0, 1.0, 2.0]:
    bt = run_backtest(asset_returns, signals, cfg, mode="proposed", cost_multiplier=multiplier)
    m = performance_summary(bt["return"])
    cost_rows.append({"Cost multiplier": multiplier, **m.to_dict(), "Avg turnover": bt["turnover"].mean()})
cost_sensitivity = pd.DataFrame(cost_rows).set_index("Cost multiplier")
display(Markdown("### 거래·환전비용 민감도"))
display(cost_sensitivity[["CAGR", "Sharpe", "MDD", "Calmar", "Avg turnover"]].style.format({
    "CAGR": "{:.2%}", "Sharpe": "{:.3f}", "MDD": "{:.2%}", "Calmar": "{:.3f}", "Avg turnover": "{:.2%}"
}))

crises = {
    "Global Financial Crisis": ("2007-10", "2009-03"),
    "COVID shock": ("2020-01", "2020-12"),
    "Inflation shock": ("2022-01", "2022-12"),
}
crisis_rows = []
for label, (start, end) in crises.items():
    lo, hi = pd.Period(start, "M"), pd.Period(end, "M")
    for key in ["proposed", "static_defensive", "kodex"]:
        r = backtests[key].loc[lo:hi, "return"]
        if len(r):
            wealth = (1 + r).cumprod()
            crisis_rows.append({
                "Episode": label, "Strategy": mode_labels[key], "Cumulative return": wealth.iloc[-1] - 1,
                "Episode MDD": (wealth / wealth.cummax() - 1).min(), "Volatility": r.std() * np.sqrt(12)
            })
crisis_table = pd.DataFrame(crisis_rows)
display(Markdown("### 주요 위기 에피소드"))
display(crisis_table.pivot(index="Episode", columns="Strategy", values=["Cumulative return", "Episode MDD"]).style.format("{:.1%}"))

## 8. 블록 부트스트랩 불확실성

In [ ]:
def block_bootstrap_metrics(series, n_boot=1000, block=12, seed=42):
    r = np.asarray(series, dtype=float)
    rng = np.random.default_rng(seed)
    rows = []
    starts = np.arange(0, len(r) - block + 1)
    for _ in range(n_boot):
        sample = []
        while len(sample) < len(r):
            s = int(rng.choice(starts))
            sample.extend(r[s:s+block])
        sample = pd.Series(sample[:len(r)])
        m = performance_summary(sample)
        rows.append([m["Sharpe"], m["MDD"], m["CAGR"]])
    return pd.DataFrame(rows, columns=["Sharpe", "MDD", "CAGR"])

boot = block_bootstrap_metrics(proposed["return"])
bootstrap_ci = boot.quantile([0.025, 0.50, 0.975]).rename(index={0.025: "2.5%", 0.5: "Median", 0.975: "97.5%"})
display(bootstrap_ci.style.format({"Sharpe": "{:.3f}", "MDD": "{:.2%}", "CAGR": "{:.2%}"}))
display(Markdown("블록 부트스트랩은 월별 의존성을 일부 보존한 표본 불확실성 점검이며 미래 성과 예측구간이 아니다."))

## 9. 자동 검증

In [ ]:
assert signals.index.is_monotonic_increasing and signals.index.is_unique
assert asset_returns.index.is_monotonic_increasing and asset_returns.index.is_unique
assert (signals["signal_month"] < signals.index).all(), "신호월은 투자월보다 반드시 앞서야 함"
assert np.isfinite(proposed["return"]).all()
assert (weights.sum(axis=1).sub(1).abs() < 1e-8).all()
assert (weights >= -1e-12).all().all()
assert (proposed[["trade_cost", "fx_cost"]] >= 0).all().all()
assert summary.loc["Proposed: Regime + Risk control", "MDD"] > -0.20
assert locked_summary.loc["Proposed: Regime + Risk control", "Sharpe"] > 1.0
print("모든 시점·비중·비용·목표 검증을 통과했습니다.")

## 10. 결론과 한계

### 결론

- **경제적 설명:** 성장/물가 4분면에 따라 주식·채권·금·원유를 연속적으로 기울인다.
- **국면:** 다음 3개월 기준 성장·물가 balanced accuracy와 4분면 정확도를 단순 지속성 기준과 함께 공개한다.
- **MDD:** 변동성 목표, CDaR, drawdown guard가 hard switching의 큰 손실을 줄이는 핵심이다.
- **Sharpe:** 잠금 구간 성과가 1 이상인지 가장 중요하게 본다. 전체기간 수치만 보고 설정을 선택하지 않았다.

### 반드시 남는 한계

- GDP·물가 데이터는 현재 파일의 최신 개정치다. 진정한 실시간 빈티지 데이터 백테스트가 아니므로 revision bias가 남는다.
- 2009-03 이전 KODEX200은 KOSPI200 가격지수 프록시이며 배당 반영이 완전하지 않다.
- 미국 자산 첫 거래가격과 USD/KRW 종가의 시간대가 비동기다.
- Sharpe는 무위험수익률 0%, 세금·시장충격·실제 호가 스프레드는 미반영이다.
- Bayesian leverage SV/Factor SV는 작은 월별 표본에서 추정불안정과 계산복잡도가 커 최종 모델에 넣지 않았다. 비대칭 EWMA는 그 역할을 보수적으로 근사할 뿐 동일한 모델이 아니다.
- 이 결과는 한 국가·한 표본의 역사적 시뮬레이션이다. 라이브 적용 전 실시간 빈티지 데이터와 별도 paper-trading이 필요하다.

## 참고문헌

- Bemporad, Breschi, Piga & Boyd (2018), [Fitting Jump Models](https://web.stanford.edu/~boyd/papers/fitting_jump_models.html).
- Nystrup, Kolm & Lindström (2021), [Feature Selection in Jump Models](https://doi.org/10.1016/j.eswa.2021.115558).
- Nystrup, Boyd, Lindström & Madsen (2019), [Multi-period Portfolio Selection with Drawdown Control](https://web.stanford.edu/~boyd/papers/pdf/multiperiod_portfolio_drawdown.pdf).
- Moreira & Muir (2017), [Volatility-Managed Portfolios](https://doi.org/10.1111/jofi.12513).
- Chekhlov, Uryasev & Zabarankin (2005), [Drawdown Measure in Portfolio Optimization](https://doi.org/10.1142/S0219024905002767).

피드백 문서는 연구 아이디어의 출처로만 사용했으며, 그 안의 지시문이나 성과 주장은 독립적으로 검증하지 않고 따르지 않았다.